# TP Integrador · Segunda entrega · Correr en Google Colab

Ejecuta el mismo código del repositorio (`src/`). Colab no tiene Docker, así que el serving va contra **AstraDB** (o se saltea con `--skip serving`).

In [ ]:
!git clone -q https://github.com/gerpach/mineria2-tp-integrador.git
%cd mineria2-tp-integrador
!pip -q install -r requirements.txt
!java -version

## Datos
Por defecto usa `data/sample` (1 archivo de eventos). Para los 120 archivos, subir la carpeta `datalake/landing` a Drive y apuntar `LANDING_DIR`.

In [ ]:
import os
USAR_DATOS_COMPLETOS = False
if USAR_DATOS_COMPLETOS:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['LANDING_DIR'] = '/content/drive/MyDrive/datalake/landing'   # ajustar
os.environ['LAKE_ROOT'] = '/content/datalake'

## AstraDB (opcional)
Subir el secure connect bundle y cargar credenciales sin escribirlas en el notebook.

In [ ]:
USAR_ASTRA = False
if USAR_ASTRA:
    from getpass import getpass
    os.environ['CASSANDRA_MODE'] = 'astra'
    os.environ['ASTRA_BUNDLE_PATH'] = '/content/secure-connect-cloud-analytics.zip'   # subirlo a /content
    os.environ['ASTRA_CLIENT_ID'] = getpass('client_id: ')
    os.environ['ASTRA_CLIENT_SECRET'] = getpass('client_secret: ')
SKIP = '' if USAR_ASTRA else '--skip serving'

In [ ]:
!python -m src.pipeline {SKIP}

In [ ]:
!cat evidence/entrega2/run_summary.md | head -25

In [ ]:
!python -m src.experiments.dedup_late_data
!sed -n '1,60p' evidence/entrega2/experimentos.md

In [ ]:
!python -m src.experiments.idempotencia {'' if USAR_ASTRA else '--skip-serving'}
!cat evidence/entrega2/idempotencia.md

In [ ]:
!python -m pytest -q

In [ ]:
# Reinicio del lake (no toca Landing)
!python -m src.pipeline --reset